# Manhwa Studio — Google Colab + Google Drive

Versão independente para renderizar com FFmpeg no ambiente do Colab e ler/gravar arquivos diretamente no Google Drive. A interface web original continua separada.

**Uso autorizado:** processe somente vídeos que você possui ou tem autorização para baixar e editar.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
WORK = Path('/content/drive/MyDrive/ManhwaStudio')
INPUT = WORK / 'entrada'
OUTPUT = WORK / 'exportados'
INPUT.mkdir(parents=True, exist_ok=True)
OUTPUT.mkdir(parents=True, exist_ok=True)
print('Entrada:', INPUT)
print('Saída:', OUTPUT)

In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg
!ffmpeg -version | head -n 2
!nvidia-smi || true

In [ ]:
!git clone --depth 1 --branch google-colab-drive https://github.com/manhwavaz1-lab/manhwa-recreator-studio.git /content/manhwa-recreator-studio
%cd /content/manhwa-recreator-studio
!python -m pip install -q yt-dlp

## 1. Colar links do vídeo e da moldura
Na interface, cole os links nos dois campos usando Ctrl+V e clique nos botões **Vincular vídeo** e **Vincular moldura**. O vídeo aceita YouTube ou Google Drive; a moldura aceita link compartilhado do Google Drive.


In [ ]:
from pathlib import Path
videos = sorted([*INPUT.glob('*.mp4'), *INPUT.glob('*.mov'), *INPUT.glob('*.mkv'), *INPUT.glob('*.webm')])
frames = sorted([*INPUT.glob('*.png'), *INPUT.glob('*.jpg'), *INPUT.glob('*.jpeg')])
print('Vídeos:', [p.name for p in videos])
print('Molduras:', [p.name for p in frames])

# Edite estes caminhos se houver mais de um arquivo:
VIDEO = str(videos[0]) if videos else ''
FRAME = str(frames[0]) if frames else ''
print('VIDEO =', VIDEO, '\\nFRAME =', FRAME)

## 2. Opcional: baixar vídeo do YouTube
Use apenas quando tiver direito/autorização para baixar o conteúdo. Deixe `YOUTUBE_URL` vazio se já colocou o MP4 no Drive.

In [ ]:
YOUTUBE_URL = ''  # cole aqui o link, se necessário
if YOUTUBE_URL.strip():
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'yt_dlp', '-f', 'bv*+ba/b', '--merge-output-format', 'mp4', '-o', str(INPUT / 'youtube_source.%(ext)s'), YOUTUBE_URL], check=True)
    downloaded = sorted(INPUT.glob('youtube_source.*'), key=lambda p:p.stat().st_mtime, reverse=True)
    VIDEO = str(downloaded[0])
print('Vídeo selecionado:', VIDEO or 'Nenhum')

## 3. Interface de renderização
Selecione o vídeo, selecione a moldura, escolha o formato e clique em renderizar. O arquivo também será salvo no Google Drive.


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import files
from pathlib import Path
import subprocess, sys, re, shutil

OUT = str(OUTPUT / 'manhwa-studio-export.mp4')
VIDEO = ''
FRAME = ''

# Cole os links com Ctrl+V. Vídeo aceita link do YouTube ou Google Drive.
video_url = widgets.Text(
    value='', placeholder='Cole aqui o link do vídeo (Ctrl+V): YouTube ou Google Drive',
    description='Link do vídeo:', layout=widgets.Layout(width='98%'),
    style={'description_width':'initial'}
)
frame_url = widgets.Text(
    value='', placeholder='Cole aqui o link da moldura PNG/JPG do Google Drive (Ctrl+V)',
    description='Link da moldura:', layout=widgets.Layout(width='98%'),
    style={'description_width':'initial'}
)
link_video_btn = widgets.Button(description='Vincular vídeo', button_style='primary', icon='link')
link_frame_btn = widgets.Button(description='Vincular moldura', button_style='primary', icon='image')
render_btn = widgets.Button(description='Renderizar vídeo', button_style='success', icon='film')
download_btn = widgets.Button(description='Baixar MP4', button_style='info', icon='download', disabled=True)
format_select = widgets.Dropdown(options=['16:9','9:16'], value='16:9', description='Formato:')
encoder_select = widgets.Dropdown(options=['auto','nvenc','qsv','amf','cpu'], value='auto', description='Encoder:')
zoom_slider = widgets.IntSlider(value=100, min=50, max=200, step=5, description='Zoom:')
crop_check = widgets.Checkbox(value=True, description='Recortar para preencher')
status = widgets.Output()

def drive_id(url):
    patterns = [r'/file/d/([a-zA-Z0-9_-]+)', r'[?&]id=([a-zA-Z0-9_-]+)']
    for pattern in patterns:
        match = re.search(pattern, url)
        if match: return match.group(1)
    return None

def download_drive_file(url, destination):
    import gdown
    file_id = drive_id(url)
    if file_id:
        result = gdown.download(id=file_id, output=str(destination), quiet=False, fuzzy=True)
    else:
        result = gdown.download(url, str(destination), quiet=False, fuzzy=True)
    if not result or not destination.exists() or destination.stat().st_size == 0:
        raise RuntimeError('Não foi possível baixar o arquivo. Verifique se o link do Drive está compartilhado com acesso por link.')
    return str(destination)

def link_video_click(_):
    global VIDEO
    with status:
        clear_output()
        url = video_url.value.strip()
        if not url:
            print('Cole o link do vídeo no campo acima usando Ctrl+V.'); return
        try:
            if 'youtube.com' in url or 'youtu.be' in url:
                subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'yt-dlp'], check=True)
                template = str(INPUT / 'video_youtube.%(ext)s')
                subprocess.run([sys.executable, '-m', 'yt_dlp', '-f', 'bv*+ba/b', '--merge-output-format', 'mp4', '-o', template, url], check=True)
                found = sorted(INPUT.glob('video_youtube.*'), key=lambda p:p.stat().st_mtime, reverse=True)
                if not found: raise RuntimeError('O download do vídeo não gerou arquivo.')
                VIDEO = str(found[0])
            else:
                subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gdown'], check=True)
                VIDEO = download_drive_file(url, INPUT / 'video_vinculado.mp4')
            print('Vídeo vinculado:', VIDEO)
        except Exception as e:
            VIDEO = ''
            print('Erro ao vincular vídeo:', e)

def link_frame_click(_):
    global FRAME
    with status:
        clear_output()
        url = frame_url.value.strip()
        if not url:
            print('Cole o link da moldura no campo acima usando Ctrl+V.'); return
        try:
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gdown'], check=True)
            # Moldura salva com extensão PNG; FFmpeg detecta o formato pelo conteúdo.
            FRAME = download_drive_file(url, INPUT / 'moldura_vinculada.png')
            print('Moldura vinculada:', FRAME)
        except Exception as e:
            FRAME = ''
            print('Erro ao vincular moldura:', e)

def render_click(_):
    with status:
        clear_output()
        if not VIDEO or not Path(VIDEO).exists():
            print('Primeiro cole o link e clique em Vincular vídeo.'); return
        if not FRAME or not Path(FRAME).exists():
            print('Cole o link da moldura e clique em Vincular moldura.'); return
        cmd=[sys.executable, 'colab/renderer.py', '--video', VIDEO,
             '--frame', FRAME, '--output', OUT, '--format', format_select.value,
             '--encoder', encoder_select.value, '--fps', '30', '--zoom', str(zoom_slider.value)]
        if crop_check.value: cmd.append('--crop')
        try:
            subprocess.run(cmd, check=True)
            download_btn.disabled = False
            print('Renderização concluída. Salvo no Google Drive:', OUT)
        except subprocess.CalledProcessError as e:
            print('Falha na renderização:', e)

def download_click(_):
    if Path(OUT).exists():
        files.download(OUT)
    else:
        with status: print('Renderize o vídeo antes de baixar.')

link_video_btn.on_click(link_video_click)
link_frame_btn.on_click(link_frame_click)
render_btn.on_click(render_click)
download_btn.on_click(download_click)

display(widgets.HTML('<h3>Vincular arquivos por link</h3><p>Cole os links usando Ctrl+V. O vídeo pode ser do YouTube ou Google Drive. A moldura deve estar no Google Drive e compartilhada por link.</p>'))
display(video_url, link_video_btn)
display(frame_url, link_frame_btn)
display(widgets.HTML('<h3>Configurações de renderização</h3>'))
display(format_select, encoder_select, zoom_slider, crop_check, render_btn, download_btn, status)
